# 2. Data Collection with Web Scraping

**Goal:** collect the same Falcon 9 launch history from Wikipedia's launch tables, as a second source. The page is fixed to the 9 June 2021 version, so the results stay the same.

In [1]:
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup

## Helper functions for the HTML table cells

In [2]:
def date_time(table_cells):
    """Return the date and time from a table cell."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]


def booster_version(table_cells):
    """Return the booster version from a table cell."""
    return "".join([bv for i, bv in enumerate(table_cells.strings) if i % 2 == 0][0:-1])


def landing_status(table_cells):
    """Return the landing status from a table cell."""
    return [i for i in table_cells.strings][0]


def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    return mass[0:mass.find("kg") + 2] if mass else 0


def extract_column_from_header(row):
    """Return a clean column name from a table header cell."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = " ".join(row.contents)
    if not column_name.strip().isdigit():
        return column_name.strip()

## Request the Wikipedia page

In [3]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

response = requests.get(static_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

## Extract the column names from the table header

In [4]:
html_tables = soup.find_all("table")
first_launch_table = html_tables[2]

column_names = []
for th in first_launch_table.find_all("th"):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
column_names

['Flight No.',
 'Date and time ( )',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome']

## Parse every launch table into a data frame

In [5]:
launch_dict = dict.fromkeys(column_names)
del launch_dict["Date and time ( )"]
for key in ["Flight No.", "Launch site", "Payload", "Payload mass", "Orbit", "Customer",
            "Launch outcome", "Version Booster", "Booster landing", "Date", "Time"]:
    launch_dict[key] = []

extracted_row = 0
for table in soup.find_all("table", "wikitable plainrowheaders collapsible"):
    for rows in table.find_all("tr"):
        # A launch row starts with a header cell that holds the flight number
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False
        row = rows.find_all("td")
        if not flag:
            continue
        extracted_row += 1
        datatimelist = date_time(row[0])
        bv = booster_version(row[1]) or row[1].a.string
        launch_dict["Flight No."].append(flight_number)
        launch_dict["Date"].append(datatimelist[0].strip(","))
        launch_dict["Time"].append(datatimelist[1])
        launch_dict["Version Booster"].append(bv)
        launch_dict["Launch site"].append(row[2].a.string)
        launch_dict["Payload"].append(row[3].a.string)
        launch_dict["Payload mass"].append(get_mass(row[4]))
        launch_dict["Orbit"].append(row[5].a.string)
        launch_dict["Customer"].append(row[6].a.string if row[6].a else row[6].text.strip())
        launch_dict["Launch outcome"].append(list(row[7].strings)[0])
        launch_dict["Booster landing"].append(landing_status(row[8]))

print(f"Rows extracted: {extracted_row}")

Rows extracted: 121


In [6]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.to_csv("../data/spacex_web_scraped.csv", index=False)
df.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10
